# Ecommify Demo en Vivo MongoDB Equipo E04



---
## Paso 1 — Conexión a MongoDB Atlas


> "Este es nuestro módulo analítico, MongoDB Atlas. Acá vive el catálogo
> de productos con esquema flexible — cada categoría puede tener atributos
> distintos sin necesidad de columnas nulas, algo que en PostgreSQL
> requeriría una solución mucho más compleja."

In [34]:
!pip install pymongo[srv] dnspython pandas --quiet

import pymongo
import datetime
import time
from google.colab import userdata

MONGODB_URI = userdata.get('MONGODB_URI')
client = pymongo.MongoClient(MONGODB_URI)
db = client["ecommify"]
products_col = db["product_catalog"]
reviews_col = db["reviews"]

print("✅ Conexión establecida.")
print(f"   product_catalog: {products_col.count_documents({}):,} documentos")
print(f"   reviews: {reviews_col.count_documents({}):,} documentos")

✅ Conexión establecida.
   product_catalog: 5,000 documentos
   reviews: 10,077 documentos


---
## Paso 2 — Eliminar índices previos


> "Voy a partir de un estado limpio, sin índices, para mostrarles el
> antes real — solo queda el índice por defecto de _id."

In [35]:
products_col.drop_indexes()
print("🧹 Índices eliminados en product_catalog.")

🧹 Índices eliminados en product_catalog.


---
## Paso 3 — Consulta ANTES del índice ESR (COLLSCAN)

> "Esta consulta busca productos de informática con más de 8 GB de RAM,
> ordenados por fecha de creación. Sin índice, MongoDB tiene que revisar
> los 5 mil documentos uno por uno — eso es un COLLSCAN."

In [36]:
query_esr = {
    "category": "informatica_acessorios",
    "attributes.ram_gb": {"$gt": 8}
}
sort_esr = [("created_at", -1)]

explain_before = products_col.find(query_esr).sort(sort_esr).explain()
stats_before = explain_before.get("executionStats", {})

print("📊 ANTES del índice:")
print(f"   Stage: {stats_before.get('executionStages', {}).get('stage')}")
print(f"   Tiempo: {stats_before.get('executionTimeMillis')} ms")
print(f"   Docs examinados: {stats_before.get('totalDocsExamined')}")

📊 ANTES del índice:
   Stage: SORT
   Tiempo: 4 ms
   Docs examinados: 5000


---
## Paso 4 — Crear el índice compuesto ESR


> "Ahora creo un índice compuesto siguiendo la regla ESR — Equality,
> Sort, Range. Primero el campo de igualdad, category. Después el de
> ordenamiento, created_at. Y al final el de rango, ram_gb. Ese orden
> no es arbitrario, es lo que maximiza el uso del índice."

In [37]:
products_col.create_index([
    ("category", pymongo.ASCENDING),
    ("created_at", pymongo.DESCENDING),
    ("attributes.ram_gb", pymongo.ASCENDING)
], name="idx_category_created_ram_esr")

print("✅ Índice compuesto ESR creado.")

✅ Índice compuesto ESR creado.


---
## Paso 5 — Consulta DESPUÉS del índice (IXSCAN)

> "Misma consulta, mismos 5 mil documentos en la colección. Miren cómo
> cambia el stage a IXSCAN y caen los documentos examinados."

In [38]:
explain_after = products_col.find(query_esr).sort(sort_esr).explain()
stats_after = explain_after.get("executionStats", {})

print("📊 DESPUÉS del índice:")
print(f"   Stage: {stats_after.get('executionStages', {}).get('stage')}")
print(f"   Tiempo: {stats_after.get('executionTimeMillis')} ms")
print(f"   Docs examinados: {stats_after.get('totalDocsExamined')}")

📊 DESPUÉS del índice:
   Stage: FETCH
   Tiempo: 1 ms
   Docs examinados: 394


---
## Paso 6 — Índice de texto full-text


> "Por último, muestro algo distinto: una búsqueda de texto libre.
> Sin índice de texto, MongoDB ni siquiera permite ejecutar esta
> consulta da error directamente."

In [41]:
query_text = {"$text": {"$search": "diseno Pro"}}

resultados = list(
    products_col.find(
        query_text,
        {
            "_id": 0,
            "name": 1,
            "description": 1,
            "score": {"$meta": "textScore"}
        }
    )
    .sort([("score", {"$meta": "textScore"})])
    .limit(5)
)

print(f"✅ Búsqueda ejecutada correctamente.")
print(f"🔎 Se encontraron {len(resultados)} resultados.\n")

for i, r in enumerate(resultados, start=1):
    print(f"{i}. {r['name']}")
    print(f"   Score de relevancia: {round(r['score'], 2)}")

    if r.get("description"):
        print(f"   Descripción: {r['description'][:100]}...")

    print("-" * 80)

✅ Búsqueda ejecutada correctamente.
🔎 Se encontraron 5 resultados.

1. Brinquedos Pro Model-4976
   Score de relevancia: 1.16
   Descripción: Excelente dispositivo de la categoria brinquedos con prestaciones de alto rendimiento y diseno moder...
--------------------------------------------------------------------------------
2. Brinquedos Pro Model-2390
   Score de relevancia: 1.16
   Descripción: Excelente dispositivo de la categoria brinquedos con prestaciones de alto rendimiento y diseno moder...
--------------------------------------------------------------------------------
3. Telefonia Pro Model-2211
   Score de relevancia: 1.16
   Descripción: Excelente dispositivo de la categoria telefonia con prestaciones de alto rendimiento y diseno modern...
--------------------------------------------------------------------------------
4. Brinquedos Pro Model-73
   Score de relevancia: 1.16
   Descripción: Excelente dispositivo de la categoria brinquedos con prestaciones de alto rendimient


> "Creo el índice de texto sobre name y description, y ahora sí
> funciona la búsqueda libre."

In [40]:
products_col.create_index([
    ("name", pymongo.TEXT),
    ("description", pymongo.TEXT)
], name="idx_text_name_description")

explain_text = products_col.find(query_text).explain()
stats_text = explain_text.get("executionStats", {})

print("✅ Índice de texto creado.")
print(f"   Ahora la búsqueda funciona: {stats_text.get('nReturned')} resultados en {stats_text.get('executionTimeMillis')} ms")

✅ Índice de texto creado.
   Ahora la búsqueda funciona: 5000 resultados en 11 ms


---
## Paso 7 — Aggregation Pipeline de 6 stages


> "Para cerrar, este es el pipeline de agregación: filtra productos,
> une con sus reviews, calcula el score promedio, y clasifica cada
> producto en un nivel de recomendación — todo en una sola consulta."

In [42]:
pipeline = [
    {"$match": {
        "category": {"$ne": "cool_stuff"},
        "created_at": {"$gte": datetime.datetime(2025, 1, 1)}
    }},
    {"$lookup": {
        "from": "reviews",
        "localField": "product_id",
        "foreignField": "product_id",
        "as": "reviews_info"
    }},
    {"$unwind": "$reviews_info"},
    {"$group": {
        "_id": "$product_id",
        "product_name": {"$first": "$name"},
        "avg_score": {"$avg": "$reviews_info.score"},
        "total_reviews": {"$sum": 1}
    }},
    {"$addFields": {
        "score_rounded": {"$round": ["$avg_score", 2]},
        "recommendation_level": {
            "$cond": {
                "if": {"$gte": ["$avg_score", 4.5]},
                "then": "Altamente Recomendado",
                "else": {
                    "$cond": {
                        "if": {"$gte": ["$avg_score", 3.5]},
                        "then": "Recomendado",
                        "else": "Neutral/Bajo"
                    }
                }
            }
        }
    }},
    {"$sort": {"avg_score": -1}}
]

start = time.time()
resultados = list(products_col.aggregate(pipeline))
elapsed = (time.time() - start) * 1000

print(f"✅ Pipeline ejecutado en {elapsed:.2f} ms — {len(resultados)} productos agregados")
for r in resultados[:3]:
    print(f"   {r['product_name'][:40]:<40} | score: {r['score_rounded']} | {r['recommendation_level']}")

✅ Pipeline ejecutado en 23837.84 ms — 4195 productos agregados
   Esporte Lazer Basic Model-1278           | score: 5.0 | Altamente Recomendado
   Esporte Lazer Basic Model-1424           | score: 5.0 | Altamente Recomendado
   Telefonia Basic Model-658                | score: 5.0 | Altamente Recomendado


---
## Cierre

> "Con esto vimos cómo los índices especializados y el aggregation
> pipeline complementan el lado transaccional que mostró Luis. Le paso
> la palabra a Brian para los resultados y el análisis comparativo."

In [ ]:
client.close()
print('✅ Demo completo. Conexión cerrada.')